# 04 - Segment Analysis

Check whether the effect is consistent across device types and countries.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config
from src.data_loader import load_cleaned, split_groups

df        = load_cleaned()
ctrl, exp = split_groups(df)
CC = config.CONTROL_COLOR
EC = config.EXPERIMENT_COLOR

plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

## 1. Conversion rate by segment

In [ ]:
def segment_table(col):
    rows = []
    for val in sorted(df[col].unique()):
        seg = df[df[col] == val]
        c   = seg[seg['group'] == 'control']['converted'].mean()
        e   = seg[seg['group'] == 'experiment']['converted'].mean()
        rows.append({col: val, 'control_%': round(c*100, 2),
                     'experiment_%': round(e*100, 2),
                     'lift_%': round((e-c)/c*100, 1)})
    return pd.DataFrame(rows)

print("By Device:")
print(segment_table('device').to_string(index=False))
print()
print("By Country:")
print(segment_table('country').to_string(index=False))

## 2. Segment bar charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Conversion Rate by Segment', fontsize=13, fontweight='bold')

for ax, col, title in zip(axes, ['device', 'country'], ['By Device', 'By Country']):
    tbl = segment_table(col)
    x   = np.arange(len(tbl))
    w   = 0.38
    ax.bar(x - w/2, tbl['control_%'],    w, label='Control',    color=CC, alpha=0.85)
    ax.bar(x + w/2, tbl['experiment_%'], w, label='Experiment', color=EC, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(tbl[col], fontsize=10)
    ax.set_ylabel('Conversion Rate (%)')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    for i, (c, e) in enumerate(zip(tbl['control_%'], tbl['experiment_%'])):
        ax.text(i-w/2, c+0.05, f'{c:.1f}%', ha='center', fontsize=7.5)
        ax.text(i+w/2, e+0.05, f'{e:.1f}%', ha='center', fontsize=7.5)

plt.tight_layout()
plt.savefig('../' + config.IMAGES_DIR + 'segment_analysis.png', bbox_inches='tight')
plt.show()

## 3. Results summary & business recommendation

In [ ]:
p_c = ctrl['converted'].mean()
p_e = exp['converted'].mean()
lift = (p_e - p_c) / p_c * 100

monthly_users = (len(df) / 14) * 30
extra_conv    = monthly_users * (p_e - p_c)
aov           = exp[exp['revenue'] > 0]['revenue'].mean()
monthly_gain  = extra_conv * aov

print("Final Summary")
print()
print(f"  Control conversion rate   : {p_c*100:.2f}%")
print(f"  Experiment conversion rate: {p_e*100:.2f}%")
print(f"  Relative lift             : +{lift:.1f}%")
print()
print("  Recommendation: Ship the new CTA design to 100% of users.")
print()
print("  Estimated monthly revenue gain  : ${:,.0f}".format(monthly_gain))
print("  Estimated annual revenue gain   : ${:,.0f}".format(monthly_gain * 12))